In [1]:
import torch
import numpy as np
from datasets import Dataset
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm
import os

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)


device: mps


In [2]:
from tablevault import tablevault

vault = tablevault.Vault(user_id="jinjin",
                            process_name="three_label_negative_decomposition_zero_shot_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [3]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding


---[ TableVault Record ]---
---[ TableVault Record ]---



In [4]:
model_name = "typeform/distilbert-base-uncased-mnli"

if device == "mps":
    clf = pipeline(
        "zero-shot-classification",
        model=model_name,
        tokenizer=model_name,
        framework="pt",
        device=torch.device("mps"),
    )
else:
    clf = pipeline(
        "zero-shot-classification",
        model=model_name,
        tokenizer=model_name,
        framework="pt",
        device=-1,
    )

candidate_labels = ["paraphrase", "contradiction", "unrelated"]
hypothesis_template = "The relationship between the two sentences is {}."

print(model_name)
print(candidate_labels)
print(hypothesis_template)



---[ TableVault Record ]---


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

typeform/distilbert-base-uncased-mnli
['paraphrase', 'contradiction', 'unrelated']
The relationship between the two sentences is {}.
---[ TableVault Record ]---



In [5]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_list(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])
texts = [f"Sentence 1: {s1}\nSentence 2: {s2}" for s1, s2 in zip(sent1, sent2)]

print("num_examples:", len(y_true))
print("positive_rate:", float(y_true.mean()))
print(texts[0])



---[ TableVault Record ]---
Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647
Sentence 1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
Sentence 2: " The foodservice pie business does not fit our long-term growth strategy .
---[ TableVault Record ]---



In [6]:
batch_size = 16
preds = []
all_ranked_labels = []
all_ranked_scores = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch_texts = texts[i:i + batch_size]
    outputs = clf(
        batch_texts,
        candidate_labels=candidate_labels,
        hypothesis_template=hypothesis_template,
        multi_label=False,
        batch_size=batch_size,
        truncation=True,
    )

    if isinstance(outputs, dict):
        outputs = [outputs]

    for out in outputs:
        labels = list(out["labels"])
        scores = [float(s) for s in out["scores"]]
        all_ranked_labels.append(labels)
        all_ranked_scores.append(scores)
        preds.append(1 if labels[0] == "paraphrase" else 0)

y_pred = np.array(preds)
print("done")



---[ TableVault Record ]---


  0%|          | 0/26 [00:00<?, ?it/s]

done
---[ TableVault Record ]---



In [7]:

vault.create_record_list("distilbert-prediction-three-label", column_names=["prediction", "ranked_labels" , "ranked_scores"])

for i in range(len(y_pred)):
    vault.append_record("distilbert-prediction-three-label", 
                        {
                            "prediction": int(y_pred[i]),
                            "ranked_labels": all_ranked_labels[i],
                            "ranked_scores": all_ranked_scores[i],
                        },
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                       }
                       )

description = "distilbert-prediction-three-label is a per-example prediction dataset generated from the GLUE MRPC validation split using the zero-shot classification model typeform/distilbert-base-uncased-mnli. For each input sentence pair from glue_mrpc_validation, it stores: prediction, a binary label where 1 means the top-ranked class was paraphrase and 0 means the top-ranked class was not paraphrase; ranked_labels, the full model ranking over the three candidate relationship labels [paraphrase, contradiction, unrelated]; and ranked_scores, the corresponding confidence scores for those labels. Each record is linked to the matching source row in glue_mrpc_validation. In this workflow, this dataset serves as the main model output table used for evaluation, error analysis, and creation of the summary metrics dataset."
embedding = get_embeddings(description)
vault.create_description("distilbert-prediction-three-label", description, embedding)

properties = {"task": "paraphrase detection", "method": "zero-shot classification", "model": "typeform/distilbert-base-uncased-mnli", "dataset_name": "distilbert-prediction-three-label", "source": "glue/mrpc", "split": "validation", "size": "408", "input_type": "sentence pair", "output_type": "predictions with ranked labels and scores", "candidate_labels": "paraphrase, contradiction, unrelated", "label_space": "binary prediction derived from three-way zero-shot labels", "evaluation_target": "mrpc label", "domain": "news", "process": "three_label_negative_decomposition_zero_shot_mrpc"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("distilbert-prediction-three-label", cat, embedding, prop)


---[ TableVault Record ]---
---[ TableVault Record ]---



In [8]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=['not_paraphrase', 'paraphrase'])

print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))



---[ TableVault Record ]---
{'accuracy': 0.6764705882352942, 'f1': 0.8006042296072508}
                precision    recall  f1-score   support

not_paraphrase       0.44      0.09      0.14       129
    paraphrase       0.69      0.95      0.80       279

      accuracy                           0.68       408
     macro avg       0.57      0.52      0.47       408
  weighted avg       0.61      0.68      0.59       408

---[ TableVault Record ]---



In [9]:
for i in range(5):
    score_map = {label: float(score) for label, score in zip(all_ranked_labels[i], all_ranked_scores[i])}
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "top_relation:", all_ranked_labels[i][0])
    print("scores:", {label: score_map[label] for label in candidate_labels})



---[ TableVault Record ]---
sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 0 top_relation: contradiction
scores: {'paraphrase': 0.004584910348057747, 'contradiction': 0.9948615431785583, 'unrelated': 0.0005535607342608273}
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0 pred: 1 top_relation: paraphrase
scores: {'paraphrase': 0.9487652778625488, 'contradiction': 0.03598674014210701, 'unrelated': 0.015247955918312073}
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the ses

In [10]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    score_map = {label: float(score) for label, score in zip(all_ranked_labels[i], all_ranked_scores[i])}
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "top_relation:", all_ranked_labels[i][0])
    print("scores:", {label: score_map[label] for label in candidate_labels})



---[ TableVault Record ]---
num_errors: 132
idx: 0
sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 0 top_relation: contradiction
scores: {'paraphrase': 0.004584910348057747, 'contradiction': 0.9948615431785583, 'unrelated': 0.0005535607342608273}
idx: 1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0 pred: 1 top_relation: paraphrase
scores: {'paraphrase': 0.9487652778625488, 'contradiction': 0.03598674014210701, 'unrelated': 0.015247955918312073}
idx: 2
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 

In [11]:
vault.create_record_list("three_label_negative_decomposition_zero_shot_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("three_label_negative_decomposition_zero_shot_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "distilbert-prediction-three-label": [0, len(ds)]
                    })

summary

description = "three_label_negative_decomposition_zero_shot_mrpc_summary is a summary-level dataset containing the final evaluation results for the notebook\u2019s zero-shot paraphrase detection experiment on the GLUE MRPC validation split. It stores one aggregate record over the full validation set, derived from comparing the true MRPC labels to predictions generated by the typeform/distilbert-base-uncased-mnli zero-shot classifier using the candidate labels paraphrase, contradiction, and unrelated.\n\nThe dataset has three fields: accuracy (float), f1 (float, for the positive paraphrase class), and classification_report (string produced by sklearn, summarizing precision, recall, f1-score, and support for the not_paraphrase and paraphrase classes).\n\nIts role in the workflow is to provide a compact experiment summary that can be saved, queried, and linked back to both the source dataset glue_mrpc_validation and the per-example prediction dataset distilbert-prediction-three-label. It is intended for quick inspection, comparison across runs, and experiment tracking rather than row-level analysis."
embedding = get_embeddings(description)
vault.create_description("three_label_negative_decomposition_zero_shot_mrpc_summary", description, embedding)

properties = {"dataset_type": "evaluation summary", "task": "paraphrase detection", "method": "zero-shot classification", "model": "typeform/distilbert-base-uncased-mnli", "source": "glue/mrpc", "input_dataset": "glue_mrpc_validation", "split": "validation", "size": "408", "label_space": "paraphrase|contradiction|unrelated", "output_fields": "accuracy|f1|classification_report", "process_name": "three_label_negative_decomposition_zero_shot_mrpc"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("three_label_negative_decomposition_zero_shot_mrpc_summary", cat, embedding, prop)



---[ TableVault Record ]---
---[ TableVault Record ]---



In [12]:
description = "This notebook runs a zero-shot paraphrase detection experiment on the GLUE MRPC validation set using the Hugging Face model typeform/distilbert-base-uncased-mnli. It frames each sentence pair as a three-way natural language inference style classification problem with candidate labels paraphrase, contradiction, and unrelated, then converts the top predicted label into the binary MRPC task by predicting paraphrase as class 1 and all other labels as class 0. The workflow loads MRPC examples from TableVault, builds sentence-pair prompts, performs batched zero-shot inference on MPS or CPU, stores per-example predictions, ranked labels, and confidence scores in TableVault, computes evaluation metrics including accuracy, F1, and a classification report, and reviews sample predictions and errors. It also writes experiment-level summaries and descriptive metadata embeddings back to TableVault so the dataset, predictions, metrics, and overall notebook process are documented and queryable." # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("three_label_negative_decomposition_zero_shot_mrpc", description, embedding)

properties = {"task": "paraphrase detection", "method": "zero-shot classification", "problem_formulation": "three-label negative decomposition", "labels": "paraphrase, contradiction, unrelated", "model": "typeform/distilbert-base-uncased-mnli", "dataset": "glue/mrpc", "dataset_split": "validation", "evaluation": "accuracy, f1-score, classification report", "framework": "transformers pipeline", "library": "PyTorch, scikit-learn, Hugging Face Datasets", "device": "mps or cpu", "prediction_output": "record-level predictions with ranked labels and scores", "tracking": "TableVault", "embedding_model": "text-embedding-3-large"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("three_label_negative_decomposition_zero_shot_mrpc", cat, embedding, prop)


---[ TableVault Record ]---
---[ TableVault Record ]---

